# Data quality anomaly inspector

Notebook Bokeh pour inspecter les findings qualite sur une serie brute. Choisir `SYMBOL`, `TABLE` et `INTERVAL` dans la cellule de configuration, puis relancer les cellules. Les anomalies recalculees depuis OHLCV sont marquees directement sur les graphiques.

In [ ]:
import sys
from pathlib import Path

for candidate in [Path.cwd(), Path.cwd() / "notebooks" / "validation"]:
    if (candidate / "data_quality_inspector.py").exists():
        sys.path.insert(0, str(candidate))
        break

import data_quality_inspector as inspector

PROJECT_ROOT = inspector.resolve_project_root()
DB_PATH = PROJECT_ROOT / "crypto_top60_h4_2022_01_2026_04.db"
QUALITY_REPORT_JSON = inspector.resolve_report_path(PROJECT_ROOT, "/tmp/top60_quality_new.json")
IMPORT_ANOMALIES_JSONL = None

# Main selection. Start with symbols that currently have useful signals:
# OMUSDT, SIRENUSDT, CVXUSDT, ICPUSDT, APTUSDT, PUMPUSDT, KLAYUSDT.
SYMBOL = "ADAUSDT"
TABLE = "futures"  # spot, futures, funding_rates
INTERVAL = "4h"

# Leave empty to show every recomputed/loaded anomaly for the selected series.
CHECKS_TO_SHOW = []
WINDOW_DAYS = 30
FOCUS_ON_FINDINGS = True

print(f"Project root: {PROJECT_ROOT}")
print(f"DB: {DB_PATH}")
print(f"Quality report: {QUALITY_REPORT_JSON}")

In [ ]:
inspector.init_notebook()

quality_findings = inspector.load_quality_findings(
    QUALITY_REPORT_JSON,
    import_anomalies_jsonl=IMPORT_ANOMALIES_JSONL,
)

inspector.finding_counts(quality_findings)

In [ ]:
inspector.show_findings_table(quality_findings)

In [ ]:
checks = CHECKS_TO_SHOW or None
inspector.render_inspector(
    DB_PATH,
    quality_findings,
    symbol=SYMBOL,
    table=TABLE,
    interval=INTERVAL,
    checks=checks,
    window_days=WINDOW_DAYS,
    focus_on_findings=FOCUS_ON_FINDINGS,
)

## Useful next selections

- `TABLE = "futures"`, `SYMBOL = "ICPUSDT"`: large time gap.
- `TABLE = "futures"`, `SYMBOL = "CVXUSDT"`: time gap and price jump.
- `TABLE = "futures"`, `SYMBOL = "PUMPUSDT"`: time gap and jump.
- `TABLE = "spot"`, `SYMBOL = "APTUSDT"`: large wick outliers.
- `TABLE = "futures"`, `SYMBOL = "SIRENUSDT"`: robust price jumps.
- `TABLE = "futures"`, `SYMBOL = "KLAYUSDT"`: stale price run.
- `TABLE = "funding_rates"`, `SYMBOL = "CVXUSDT"`: funding gap and extremes.